In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import r2_score, root_mean_squared_error

In [4]:
df = pd.read_csv('../data/processed/rental_processed.csv')

print(df.shape)
print(df.head())

(7691, 5482)
        locality    area  beds  bathrooms  balconies  area_rate      rent  \
0  Goregaon East   897.0     2          2          0      134.0  120000.0   
1          Powai   490.0     1          1          0       82.0   40000.0   
2        Mundhwa   550.0     1          1          0       22.0   12000.0   
3         Hingna  1000.0     2          2          0        8.0    8000.0   
4      Mira Road   595.0     1          1          0       25.0   15000.0   

   house_type_1 BHK Flat for Rent in 7th Heaven, Dhanori, Pune  \
0                                              False             
1                                              False             
2                                              False             
3                                              False             
4                                              False             

   house_type_1 BHK Flat for Rent in Aaditya Glory II, Godhani, Nagpur  \
0                                              False 

In [5]:
# Target variable
y = df['rent']

# Features
X = df.drop(columns=['rent', 'locality'])

print("Features:", X.shape)
print("Target:", y.shape)

Features: (7691, 5480)
Target: (7691,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)

(6152, 5480) (1539, 5480)


In [7]:
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_r2 = r2_score(y_test, lr_pred)
lr_rmse = root_mean_squared_error(y_test, lr_pred)

print("Linear Regression")
print("R2   :", round(lr_r2, 4))
print("RMSE :", round(lr_rmse, 2))


Linear Regression
R2   : 0.6637
RMSE : 50199.54


In [8]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = root_mean_squared_error(y_test, rf_pred)

print("Random Forest")
print("R2   :", round(rf_r2, 4))
print("RMSE :", round(rf_rmse, 2))

Random Forest
R2   : 0.9628
RMSE : 16695.53


In [9]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

xgb_r2 = r2_score(y_test, xgb_pred)
xgb_rmse = root_mean_squared_error(y_test, xgb_pred)

print("XGBoost")
print("R2   :", round(xgb_r2, 4))
print("RMSE :", round(xgb_rmse, 2))

XGBoost
R2   : 0.9642
RMSE : 16381.73


In [10]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'R2': [lr_r2, rf_r2, xgb_r2],
    'RMSE': [lr_rmse, rf_rmse, xgb_rmse]
})

results = results.sort_values(by='R2', ascending=False)

print(results)

               Model        R2          RMSE
2            XGBoost  0.964190  16381.725245
1      Random Forest  0.962805  16695.528678
0  Linear Regression  0.663737  50199.542970


In [11]:
import pickle

with open('../models/rent_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

print('Model saved successfully!')

Model saved successfully!


In [12]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(xgb, "../models/xgb_model.pkl")
joblib.dump(X.columns.tolist(), "../models/feature_columns.pkl")

print("=================================")
print("MODEL SAVED SUCCESSFULLY")
print("=================================")
print("Model: ../models/xgb_model.pkl")
print("Features: ../models/feature_columns.pkl")
print("Number of features:", len(X.columns))

MODEL SAVED SUCCESSFULLY
Model: ../models/xgb_model.pkl
Features: ../models/feature_columns.pkl
Number of features: 5480
